# GHZ Challenge: Honest Noisy Simulator Solution

Builds an N-qubit GHZ state with a binary fan-out tree (depth 1 + ceil(log2 N), no SWAPs), applies optional readout mitigation, scans N for the best score, and exports QASM.

Check `FIDELITY_EXPONENT` and `SEARCH_RANGE` against the grader before running.

In [ ]:
# Run once if Qiskit is missing
# %pip install -q "qiskit>=1.2" "qiskit-aer>=0.15"

import math
from qiskit import QuantumCircuit, qasm2
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

# --- Configuration ---
OPTIMAL_N         = None            # set an int to skip the scan
SHOTS             = 8192            # final GHZ run
CAL_SHOTS         = 4096            # readout calibration
SCAN_SHOTS        = 4096            # N-scan
SEARCH_RANGE      = range(60, 157)  # N values to scan
FIDELITY_EXPONENT = 1.5             # per challenge spec
SEED              = 42

## 1. Noise model and simulator
Depolarizing errors: 0.1% on 1-qubit gates, 1.5% on 2-qubit gates. The stabilizer method is used because the circuit is Clifford with Pauli noise, which lets it run at large N.

In [ ]:
def make_noise_model() -> NoiseModel:
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(0.001, 1), ['x', 'sx', 'rz', 'h'])
    nm.add_all_qubit_quantum_error(depolarizing_error(0.015, 2), ['cx', 'swap'])
    return nm

sim = AerSimulator(noise_model=make_noise_model(), method='stabilizer', seed_simulator=SEED)

## 2. GHZ circuit (binary fan-out tree)
N-1 CNOTs, depth 1 + ceil(log2 N), no SWAPs.

In [ ]:
def build_ghz(n: int) -> QuantumCircuit:
    qc = QuantumCircuit(n, n, name=f'GHZ-{n}-fan-out')
    qc.h(0)
    entangled = 1
    while entangled < n:
        new = min(entangled, n - entangled)
        for i in range(new):
            qc.cx(i, entangled + i)
        entangled += new
    qc.measure(range(n), range(n))
    return qc

demo = build_ghz(8)
print('Depth (excl. measure):', demo.depth() - 1, '| D_min:', 1 + math.ceil(math.log2(8)))
print('Ops:', dict(demo.count_ops()))

## 3. Readout calibration and mitigation
Estimates P(0|0) and P(1|1), then corrects the GHZ correlation under a tensor-product readout model. The noise model has no readout error, so this should return values near 1.

In [ ]:
def calibrate_readout(sim, shots):
    qc0 = QuantumCircuit(1, 1); qc0.measure(0, 0)
    qc1 = QuantumCircuit(1, 1); qc1.x(0); qc1.measure(0, 0)
    c0 = sim.run(qc0, shots=shots).result().get_counts()
    c1 = sim.run(qc1, shots=shots).result().get_counts()
    return c0.get('0', 0) / shots, c1.get('1', 0) / shots

def mitigate_correlation(counts, N, p0g0, p1g1):
    total = sum(counts.values())
    raw_p0 = counts.get('0' * N, 0) / total
    raw_p1 = counts.get('1' * N, 0) / total
    att0, att1 = p0g0 ** N, p1g1 ** N
    corr_p0 = raw_p0 / att0 if att0 > 1e-15 else raw_p0
    corr_p1 = raw_p1 / att1 if att1 > 1e-15 else raw_p1
    s = corr_p0 + corr_p1
    if s > 1.0:
        corr_p0, corr_p1 = corr_p0 / s, corr_p1 / s
    return min(corr_p0 + corr_p1, 1.0)

p0g0, p1g1 = calibrate_readout(sim, CAL_SHOTS)
print(f'P(read 0 | prep 0) = {p0g0:.4f}')
print(f'P(read 1 | prep 1) = {p1g1:.4f}')

## 4. Score calculation

In [ ]:
def compute_score(circuit, C, verbose=False):
    N = circuit.num_qubits
    ops = dict(circuit.count_ops())
    gate_depth = circuit.depth() - (1 if ops.get('measure', 0) > 0 else 0)
    swap_count = ops.get('swap', 0)

    C_rand = 1.0 / (2.0 ** (N - 1))
    base = 1000.0 + 50.0 * (N ** 2 - 4)
    D_min = 1 + math.ceil(math.log2(N))

    if C <= C_rand:
        return 0

    norm = (C - C_rand) / (1.0 - C_rand)
    fid = norm ** FIDELITY_EXPONENT
    swap_pen = 1.0 / (1.0 + 0.25 * swap_count)
    depth_eff = D_min / max(gate_depth, D_min)
    total = round(base * fid * swap_pen * depth_eff)

    if verbose:
        w = 54
        print('=' * w)
        print(f'  SCORE BREAKDOWN -- {N}-qubit GHZ')
        print('=' * w)
        print(f'  Base potential          {base:>18,.0f} pts')
        print(f'  Correlation C           {C:>18.6f}')
        print(f'  Random baseline C_rand  {C_rand:>18.2e}')
        print(f'  Normalised fidelity     {norm:>18.6f}')
        print(f'  Fidelity factor (^{FIDELITY_EXPONENT})   {fid:>18.6f}')
        print(f'  SWAP count              {swap_count:>18d}')
        print(f'  Gate depth (actual)     {gate_depth:>18d}')
        print(f'  Gate depth (D_min)      {D_min:>18d}')
        print(f'  Depth efficiency        {depth_eff:>18.6f}')
        print('-' * w)
        print(f'  TOTAL SCORE             {total:>18,} pts')
        print('=' * w)
    return total

## 5. Optimal N search
Scans each N and keeps the best mitigated score. With a single shot per N the argmax is noisy, so treat the chosen N as provisional.

In [ ]:
def find_optimal_n(sim, p0g0, p1g1, search_range, shots):
    print(f"{'N':>5}  {'C_raw':>7}  {'C_mit':>7}  {'score':>9}")
    print('-' * 36)
    best_score, best_n = 0, search_range[0]
    for N in search_range:
        qc = build_ghz(N)
        counts = sim.run(qc, shots=shots).result().get_counts()
        total = sum(counts.values())
        C_raw = (counts.get('0' * N, 0) + counts.get('1' * N, 0)) / total
        C_mit = mitigate_correlation(counts, N, p0g0, p1g1)
        score = compute_score(qc, C_mit)
        marker = '  <- best' if score > best_score else ''
        print(f'{N:>5}  {C_raw:>7.4f}  {C_mit:>7.4f}  {score:>9,}{marker}')
        if score > best_score:
            best_score, best_n = score, N
    print(f'\nOptimal N = {best_n} (score {best_score:,})')
    return best_n, best_score

if OPTIMAL_N is None:
    N, _ = find_optimal_n(sim, p0g0, p1g1, SEARCH_RANGE, SCAN_SHOTS)
else:
    N = OPTIMAL_N
    print(f'Using pre-set OPTIMAL_N = {N}')

## 6. Final run and QASM export

In [ ]:
qc = build_ghz(N)
counts = sim.run(qc, shots=SHOTS).result().get_counts()
total = sum(counts.values())
C_raw = (counts.get('0' * N, 0) + counts.get('1' * N, 0)) / total
C_mit = mitigate_correlation(counts, N, p0g0, p1g1)

print(f'N = {N}, shots = {SHOTS}')
print(f'C (raw)       = {C_raw:.6f}')
print(f'C (mitigated) = {C_mit:.6f}')
score = compute_score(qc, C_mit, verbose=True)

qasm_file = f'ghz_{N}_honest.qasm'
with open(qasm_file, 'w') as f:
    f.write(qasm2.dumps(qc))
print(f'QASM saved: {qasm_file}')